# A/B Test Analysis — Landing Page Conversion
**Hypothesis:** The new landing page increases conversion rate  
**Test:** Two-proportion z-test + chi-square test  
**Significance level:** α = 0.05

## 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print("All imports successful")

## 2 — Load Data

In [ ]:
df = pd.read_csv('ab_data.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 3 — Data Quality Checks

In [ ]:
# ── 3a. Nulls ──────────────────────────────────────────────
print("=== NULL COUNTS ===")
print(df.isnull().sum())
print()

In [ ]:
# ── 3b. Duplicate user IDs ─────────────────────────────────
# A user should only appear once in an A/B test.
# Duplicates skew conversion rates if the same user converts twice.
total_rows   = len(df)
unique_users = df['user_id'].nunique()
duplicates   = total_rows - unique_users

print(f"=== DUPLICATE CHECK ===")
print(f"Total rows:    {total_rows:,}")
print(f"Unique users:  {unique_users:,}")
print(f"Duplicates:    {duplicates:,}")
print()

In [ ]:
# ── 3c. Group / page assignment breakdown ──────────────────
# Expect: control = old_page only, treatment = new_page only.
# Any other combination is a mismatch — users shown the wrong page.
print("=== GROUP / PAGE ASSIGNMENTS ===")
print(df.groupby(['group', 'landing_page']).size().reset_index(name='count'))
print()

# Count mismatches explicitly
mismatches = df[
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'control')   & (df['landing_page'] == 'new_page'))
]
print(f"Mismatched rows: {len(mismatches):,}")

## 4 — Clean Data

In [ ]:
# ── 4a. Remove mismatched assignments ──────────────────────
# Users shown the wrong page received a different treatment
# than intended — including them would contaminate both groups.
df_clean = df[
    ((df['group'] == 'control')   & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))
].copy()

# ── 4b. Remove duplicate user IDs ─────────────────────────
# Keep only the first occurrence per user (earliest timestamp).
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean = df_clean.sort_values('timestamp').drop_duplicates(subset='user_id', keep='first')

print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning:  {len(df_clean):,}")
print(f"Rows removed:         {len(df) - len(df_clean):,}")

## 5 — Hypothesis Definition

In [ ]:
# Formally define the hypothesis before running any test.
# This prevents p-hacking — decide the test type BEFORE seeing results.

ALPHA = 0.05   # Significance level: 5% chance of false positive

print("=" * 55)
print("HYPOTHESIS")
print("=" * 55)
print("H₀ (null):        p_treatment = p_control")
print("                  New page has NO effect on conversion")
print()
print("H₁ (alternative): p_treatment > p_control")
print("                  New page INCREASES conversion rate")
print()
print(f"Significance (α): {ALPHA}")
print("Test type:        One-tailed two-proportion z-test")
print("                  + Chi-square test (independence check)")
print("=" * 55)

## 6 — Descriptive Statistics

In [ ]:
# Split into control and treatment groups
control   = df_clean[df_clean['group'] == 'control']
treatment = df_clean[df_clean['group'] == 'treatment']

# Core counts
n_ctrl    = len(control)
n_trt     = len(treatment)
conv_ctrl = control['converted'].sum()
conv_trt  = treatment['converted'].sum()

# Rates
rate_ctrl = conv_ctrl / n_ctrl
rate_trt  = conv_trt  / n_trt
uplift    = (rate_trt - rate_ctrl) / rate_ctrl * 100

summary = pd.DataFrame({
    'Group':           ['Control', 'Treatment'],
    'Users':           [f"{n_ctrl:,}",    f"{n_trt:,}"],
    'Conversions':     [f"{conv_ctrl:,}", f"{conv_trt:,}"],
    'Conversion Rate': [f"{rate_ctrl:.4%}", f"{rate_trt:.4%}"]
})

print("=== DESCRIPTIVE STATISTICS ===")
print(summary.to_string(index=False))
print(f"\nUplift: {uplift:+.2f}%")

## 7 — Sample Ratio Mismatch Check

In [ ]:
# A well-run experiment should assign users 50/50.
# A large imbalance (>55/45) suggests a flawed randomisation process
# and would invalidate the experiment regardless of p-value.

pct_ctrl = n_ctrl / (n_ctrl + n_trt) * 100
pct_trt  = n_trt  / (n_ctrl + n_trt) * 100

print(f"Control:   {n_ctrl:,} users ({pct_ctrl:.1f}%)")
print(f"Treatment: {n_trt:,} users ({pct_trt:.1f}%)")

if abs(pct_ctrl - pct_trt) > 5:
    print("\n⚠️  WARNING: Sample ratio mismatch detected. Investigate randomisation.")
else:
    print("\n✓  Sample ratio looks balanced.")

## 8 — Two-Proportion Z-Test

In [ ]:
# The z-test is appropriate here because:
#   - Binary outcome (converted: 0 or 1)
#   - Large sample size (n > 30 per group — Central Limit Theorem applies)
#   - Two independent groups
#
# alternative='larger' = one-tailed test (treatment > control)

count = np.array([conv_trt, conv_ctrl])   # treatment first
nobs  = np.array([n_trt,    n_ctrl])

z_stat, p_value = proportions_ztest(count, nobs, alternative='larger')

print("=== TWO-PROPORTION Z-TEST ===")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value:     {p_value:.4f}")
print()

if p_value < ALPHA:
    print(f"✓  p={p_value:.4f} < α={ALPHA} → REJECT H₀")
    print("   The new page SIGNIFICANTLY increases conversion rate.")
else:
    print(f"✗  p={p_value:.4f} ≥ α={ALPHA} → FAIL TO REJECT H₀")
    print("   No statistically significant improvement detected.")

## 9 — Confidence Intervals

In [ ]:
# 95% confidence intervals for each group's true conversion rate.
# If the CIs overlap substantially, the difference is unlikely to be real.
# method='wilson' is preferred over 'normal' for proportions —
# it handles edge cases better and is more accurate at small samples.

ci_ctrl = proportion_confint(conv_ctrl, n_ctrl, alpha=0.05, method='wilson')
ci_trt  = proportion_confint(conv_trt,  n_trt,  alpha=0.05, method='wilson')

print("=== 95% CONFIDENCE INTERVALS ===")
print(f"Control:   ({ci_ctrl[0]:.4%}, {ci_ctrl[1]:.4%})  midpoint: {rate_ctrl:.4%}")
print(f"Treatment: ({ci_trt[0]:.4%},  {ci_trt[1]:.4%})   midpoint: {rate_trt:.4%}")
print()

# Overlap check
overlap = ci_trt[0] < ci_ctrl[1]
if overlap:
    print("CIs overlap — consistent with no significant difference.")
else:
    print("CIs do not overlap — strong evidence of a real difference.")

## 10 — Chi-Square Test

In [ ]:
# Chi-square tests whether conversion and group assignment are independent.
# It is a two-tailed test — it detects ANY difference, not just improvement.
# Use it as a cross-check alongside the z-test.
# Both tests should agree — if they diverge, investigate further.

# Contingency table:
#               Converted   Not Converted
# Control       conv_ctrl   n_ctrl - conv_ctrl
# Treatment     conv_trt    n_trt  - conv_trt

contingency = np.array([
    [conv_ctrl, n_ctrl - conv_ctrl],
    [conv_trt,  n_trt  - conv_trt]
])

chi2, p_chi2, dof, expected = chi2_contingency(contingency)

print("=== CHI-SQUARE TEST ===")
print(f"Contingency table:")
print(pd.DataFrame(
    contingency,
    index=['Control', 'Treatment'],
    columns=['Converted', 'Not Converted']
))
print()
print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom:   {dof}")
print(f"P-value (two-tailed): {p_chi2:.4f}")
print()

if p_chi2 < ALPHA:
    print(f"✓  p={p_chi2:.4f} < α={ALPHA} → Conversion and group ARE associated.")
else:
    print(f"✗  p={p_chi2:.4f} ≥ α={ALPHA} → No significant association detected.")

## 11 — Results Summary

In [ ]:
print("=" * 55)
print("RESULTS SUMMARY")
print("=" * 55)
print(f"Control conversion rate:    {rate_ctrl:.4%}")
print(f"Treatment conversion rate:  {rate_trt:.4%}")
print(f"Uplift:                     {uplift:+.2f}%")
print()
print(f"Z-test   p-value:  {p_value:.4f}")
print(f"Chi-sq   p-value:  {p_chi2:.4f}")
print(f"Significance (α):  {ALPHA}")
print()
print(f"Control   95% CI:  ({ci_ctrl[0]:.4%}, {ci_ctrl[1]:.4%})")
print(f"Treatment 95% CI:  ({ci_trt[0]:.4%},  {ci_trt[1]:.4%})")
print()

decision = "REJECT H₀" if p_value < ALPHA else "FAIL TO REJECT H₀"
print(f"Decision: {decision}")
print("=" * 55)

## 12 — Visualisations

In [ ]:
# ── Plot 1: Conversion rate bar chart with CI error bars ───

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('A/B Test — Landing Page Conversion Analysis', fontsize=13, fontweight='bold', y=1.01)

groups    = ['Control', 'Treatment']
rates     = [rate_ctrl, rate_trt]
colors    = ['#6C8EBF', '#82B366']
ci_lower  = [rate_ctrl - ci_ctrl[0], rate_trt - ci_trt[0]]
ci_upper  = [ci_ctrl[1] - rate_ctrl, ci_trt[1]  - rate_trt]
yerr      = [ci_lower, ci_upper]

bars = axes[0].bar(groups, rates, color=colors, width=0.5,
                   yerr=np.array(yerr), capsize=6, error_kw={'linewidth': 1.5})
axes[0].set_title('Conversion Rate by Group\n(with 95% CI)', fontsize=11)
axes[0].set_ylabel('Conversion Rate')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
axes[0].set_ylim(0, max(rates) * 1.3)

# Label bars
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.001,
                 f'{rate:.3%}', ha='center', va='bottom', fontsize=10)

# p-value annotation
sig_label = f'p = {p_value:.4f}'
sig_label += ' ✓ Significant' if p_value < ALPHA else ' ✗ Not significant'
axes[0].text(0.5, 0.95, sig_label, transform=axes[0].transAxes,
             ha='center', va='top', fontsize=9,
             color='green' if p_value < ALPHA else 'red')

plt.tight_layout()
print("Plot 1 complete")

In [ ]:
# ── Plot 2: Daily conversion rate over time ────────────────
# Checks for novelty effect — if treatment spikes early then declines,
# the result may not be sustainable.

df_clean['date'] = df_clean['timestamp'].dt.date

daily = (
    df_clean.groupby(['date', 'group'])['converted']
    .mean()
    .reset_index()
    .rename(columns={'converted': 'conv_rate'})
)

fig, ax = plt.subplots(figsize=(12, 4))

for grp, color in zip(['control', 'treatment'], ['#6C8EBF', '#82B366']):
    subset = daily[daily['group'] == grp]
    ax.plot(subset['date'], subset['conv_rate'],
            label=grp.capitalize(), color=color, linewidth=1.8, marker='o', markersize=3)

ax.set_title('Daily Conversion Rate Over Time\n(check for novelty effects)', fontsize=11)
ax.set_xlabel('Date')
ax.set_ylabel('Conversion Rate')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('plot_daily_conversion.png', bbox_inches='tight', dpi=150)
plt.show()
print("Plot 2 saved: plot_daily_conversion.png")

## 13 — Export Results for Power BI

In [ ]:
# ── Export 1: Summary stats ────────────────────────────────
summary_export = pd.DataFrame({
    'group':               ['control',  'treatment'],
    'total_users':         [n_ctrl,     n_trt],
    'conversions':         [conv_ctrl,  conv_trt],
    'conversion_rate':     [rate_ctrl,  rate_trt],
    'ci_lower':            [ci_ctrl[0], ci_trt[0]],
    'ci_upper':            [ci_ctrl[1], ci_trt[1]],
})
summary_export.to_csv('output_summary.csv', index=False)
print("output_summary.csv exported")

# ── Export 2: Statistical test results ────────────────────
stats_export = pd.DataFrame([{
    'uplift_pct':          round(uplift, 4),
    'z_statistic':         round(z_stat, 4),
    'z_pvalue':            round(p_value, 4),
    'chi2_statistic':      round(chi2, 4),
    'chi2_pvalue':         round(p_chi2, 4),
    'alpha':               ALPHA,
    'significant':         p_value < ALPHA,
    'decision':            'Reject H0' if p_value < ALPHA else 'Fail to reject H0'
}])
stats_export.to_csv('output_stats.csv', index=False)
print("output_stats.csv exported")

# ── Export 3: Daily time series ───────────────────────────
daily.to_csv('output_daily.csv', index=False)
print("output_daily.csv exported")

# ── Export 4: Full clean dataset ──────────────────────────
df_clean.to_csv('output_clean.csv', index=False)
print("output_clean.csv exported")

print("\nAll exports complete — load these 4 files into Power BI")